In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
SEED = 42
N_SAMPLES = 6000
OUTPUT_PATH = os.path.join("..", "..", "steel+plates+faults", "steel_defect_data.csv")

In [4]:
IDEAL_CARBON_PCT = 0.40
TARGET_MANGANESE_PCT = 0.90
IDEAL_FURNACE_TEMP_C = 1600
IDEAL_COOLING_RATE_CPS = 14

# Sigmoid shift: controls overall defect rate. Tuned by trial and error
# to land the base defect rate around 20-25%, which is realistic for a
# steel process running with normal (not excellent, not terrible) control.
SIGMOID_SHIFT = 1.2
NOISE_STD = 0.6


def generate_features(n, seed):
    rng_state = np.random.RandomState(seed)

    carbon_pct = rng_state.normal(0.40, 0.12, n).clip(0.05, 1.2)
    manganese_pct = rng_state.normal(0.90, 0.30, n).clip(0.10, 2.0)
    furnace_temp_C = rng_state.normal(1600, 55, n).clip(1450, 1750)
    rolling_speed_mps = rng_state.normal(20, 6, n).clip(5, 40)
    cooling_rate_Cps = rng_state.normal(15, 6, n).clip(2, 40)

    return carbon_pct, manganese_pct, furnace_temp_C, rolling_speed_mps, cooling_rate_Cps

In [5]:
def compute_risk(carbon, manganese, furnace_temp, rolling_speed, cooling_rate):

    # Carbon: U-shaped. Too little = weak steel, too much = brittle steel.
    carbon_risk = (carbon - IDEAL_CARBON_PCT) ** 2

    # Manganese: one-directional. Binds sulfur, protects the steel --
    # only a DEFICIT is risky, no penalty for having plenty.
    manganese_risk = np.maximum(0, (TARGET_MANGANESE_PCT - manganese))

    # Furnace temp: U-shaped. Too cold = impurities not refined out,
    # too hot = grain growth/warping.
    temp_risk = ((furnace_temp - IDEAL_FURNACE_TEMP_C) / 50) ** 2

    # Cooling rate: U-shaped. Too slow = poor microstructure, too fast =
    # internal stress/cracking from thermal shock.
    cooling_risk = ((cooling_rate - IDEAL_COOLING_RATE_CPS) / 10) ** 2

    # Interaction: rolling too fast while the steel is too cold causes
    # surface cracking. Neither factor alone is dangerous -- multiplying
    # two max(0, ...) terms means risk only appears when BOTH are present.
    fastness = np.maximum(0, (rolling_speed - 20) / 10)
    coldness = np.maximum(0, (IDEAL_FURNACE_TEMP_C - furnace_temp) / 50)
    interaction_risk = fastness * coldness

    total_risk = carbon_risk + manganese_risk + temp_risk + cooling_risk + interaction_risk
    return total_risk

In [6]:
def risk_to_label(risk_raw, seed):
    
    rng_state = np.random.RandomState(seed + 1)  # different stream than feature gen

    risk_z = (risk_raw - risk_raw.mean()) / risk_raw.std()
    risk_z_noisy = risk_z + rng_state.normal(0, NOISE_STD, len(risk_raw))

    prob_defect = 1 / (1 + np.exp(-(risk_z_noisy - SIGMOID_SHIFT)))
    defect = rng_state.binomial(1, prob_defect)

    return defect, prob_defect

In [7]:
def main():
    carbon, manganese, furnace_temp, rolling_speed, cooling_rate = generate_features(N_SAMPLES, SEED)

    risk_raw = compute_risk(carbon, manganese, furnace_temp, rolling_speed, cooling_rate)
    defect, prob_defect = risk_to_label(risk_raw, SEED)

    df = pd.DataFrame({
        "batch_id": [f"B{100000 + i}" for i in range(N_SAMPLES)],
        "carbon_pct": carbon.round(3),
        "manganese_pct": manganese.round(3),
        "furnace_temp_C": furnace_temp.round(1),
        "rolling_speed_mps": rolling_speed.round(2),
        "cooling_rate_Cps": cooling_rate.round(2),
        "defect": defect,
    })

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
    print("\nDefect rate:")
    print(df["defect"].value_counts(normalize=True).round(3))
    print("\nFeature summary:")
    print(df.describe().round(3))


if __name__ == "__main__":
    main()

Saved 6000 rows to ..\..\steel+plates+faults\steel_defect_data.csv

Defect rate:
defect
0    0.735
1    0.265
Name: proportion, dtype: float64

Feature summary:
       carbon_pct  manganese_pct  furnace_temp_C  rolling_speed_mps  \
count    6000.000       6000.000        6000.000           6000.000   
mean        0.400          0.898        1601.438             19.970   
std         0.120          0.300          54.892              5.926   
min         0.050          0.100        1450.000              5.000   
25%         0.320          0.696        1563.800             15.810   
50%         0.400          0.894        1601.750             19.950   
75%         0.479          1.101        1638.900             24.050   
max         0.871          1.959        1750.000             40.000   

       cooling_rate_Cps    defect  
count          6000.000  6000.000  
mean             14.972     0.265  
std               5.877     0.441  
min               2.000     0.000  
25%              10